%md
# 04e · Oro: tablas de servicio del tablero

Cada vista del tablero lee una tabla hecha para ella: el dataset queda como
`SELECT … WHERE fecha = :fecha`, sin lógica de negocio. Así la lógica queda en el contrato de oro
(requisito 5), `ow_consumidores` la usa sin acceso a `silver`, y Delta Sharing la entrega hecha (bono).

| Tabla | Grano | Vista del tablero |
|---|---|---|
| `dim_puerto`, `dim_tipo_buque` | puerto / código | Nombres y coordenadas para quien solo lee oro |
| `kpi_dia` | día | Contadores |
| `trafico_celda_dia` | celda H3 (`RES_MAPA`) × día | Mapa de intensidad (buques distintos reales del día) |
| `trafico_hora` | día × hora × categoría | Buques por hora (UTC) |
| `puerto_dia` | puerto × día | Ranking y mapa de puertos; `arribos` es la variable objetivo de ML |
| `buques_en_puerto_dia` | buque × puerto × día | Tabla de buques en puerto |
| `anomalias_buque_dia` | buque × tipo × día | Detalle y contador de anomalías |

In [0]:
from pyspark.databricks.sql import functions as dbf
from pyspark.sql import functions as f

import comun
import config

spark = config.obtener_spark()
spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOGO = config.leer_parametro("catalogo", config.CATALOGO_DEFECTO)
RECALCULAR_TODO = config.leer_parametro("recalcular_todo", "false").lower() == "true"
cfg = config.Config(CATALOGO)

S, G = (lambda n: cfg.tabla("silver", n)), (lambda n: cfg.tabla("gold", n))
T_METRICAS, T_SILVER, T_PUERTOS_S, T_TIPOS_S = S("calidad_metricas"), S("ais_posiciones"), S("dim_puertos"), S("dim_tipo_buque")
T_PUNTOS, T_DIARIAS, T_PRESENCIA, T_ANOMALIAS = G("trayectorias_puntos"), G("trayectorias_diarias"), G("presencia_puertos"), G("anomalias_dia")
T_DIM_PUERTO, T_DIM_TIPO = G("dim_puerto"), G("dim_tipo_buque")
T_KPI, T_CELDA, T_HORA = G("kpi_dia"), G("trafico_celda_dia"), G("trafico_hora")
T_PUERTO_DIA, T_BUQUES_PUERTO, T_ANOM_BUQUE = G("puerto_dia"), G("buques_en_puerto_dia"), G("anomalias_buque_dia")
TABLAS_SERVICIO = [T_KPI, T_CELDA, T_HORA, T_PUERTO_DIA, T_BUQUES_PUERTO, T_ANOM_BUQUE]
print(f"Catálogo: {cfg.catalogo} · resolución del mapa: H3 {config.RES_MAPA}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T_DIM_PUERTO}
COMMENT 'Puertos (World Port Index), copia de silver.dim_puertos para quien solo lee oro o recibe el share.'
AS SELECT wpi_id, nombre_puerto, pais, lat, lon, tamano_puerto, tipo_puerto, _version_fuente FROM {T_PUERTOS_S}
""")
spark.sql(f"""
CREATE OR REPLACE TABLE {T_DIM_TIPO}
COMMENT 'Tipos de buque AIS y su categoría, copia de silver.dim_tipo_buque para quien solo lee oro.'
AS SELECT vessel_type, descripcion, categoria, _version_fuente FROM {T_TIPOS_S}
""")
print(f"{T_DIM_PUERTO}: {spark.table(T_DIM_PUERTO).count():,} · {T_DIM_TIPO}: {spark.table(T_DIM_TIPO).count():,}")

In [0]:
contratos = {
    T_KPI: """fecha DATE NOT NULL COMMENT 'Fecha UTC',
        buques_activos BIGINT COMMENT 'MMSI distintos con al menos un viaje',
        viajes BIGINT COMMENT 'Viajes (trayectorias_diarias)',
        millas DOUBLE COMMENT 'Millas náuticas navegadas (candado cinemático aplicado)',
        arribos BIGINT COMMENT 'Arribos a puerto (presencia_puertos)',
        buques_en_puerto BIGINT COMMENT 'Buques distintos presentes en algún puerto',
        buques_con_anomalias BIGINT COMMENT 'MMSI distintos con al menos una anomalía',
        filas_recibidas BIGINT COMMENT 'Filas que llegaron (calidad_metricas)',
        filas_cuarentena BIGINT COMMENT 'Filas en cuarentena',
        pct_cuarentena DOUBLE COMMENT 'filas_cuarentena / filas_recibidas × 100',
        _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_kpi_dia PRIMARY KEY (fecha)""",
    T_CELDA: """fecha DATE NOT NULL, h3_celda BIGINT NOT NULL COMMENT 'Celda H3 de resolución RES_MAPA',
        lat DOUBLE COMMENT 'Latitud del centro de la celda', lon DOUBLE COMMENT 'Longitud del centro de la celda',
        buques BIGINT COMMENT 'MMSI distintos en la celda durante el día',
        posiciones BIGINT COMMENT 'Reportes en la celda durante el día',
        intensidad DOUBLE COMMENT 'LOG10(posiciones): escala de color del mapa',
        sog_promedio DOUBLE, _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_trafico_celda_dia PRIMARY KEY (fecha, h3_celda)""",
    T_HORA: """fecha DATE NOT NULL, hora INT NOT NULL COMMENT 'Hora UTC (0-23)',
        categoria STRING NOT NULL COMMENT 'Categoría de buque o Desconocido',
        buques BIGINT COMMENT 'MMSI distintos en la hora, en todo el país',
        posiciones BIGINT, _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_trafico_hora PRIMARY KEY (fecha, hora, categoria)""",
    T_PUERTO_DIA: """fecha DATE NOT NULL, wpi_id INT NOT NULL, nombre_puerto STRING, pais STRING,
        lat DOUBLE, lon DOUBLE,
        arribos BIGINT COMMENT 'Arribos del día (variable objetivo del pronóstico de ML)',
        buques_presentes BIGINT COMMENT 'Buques distintos con presencia en el puerto ese día',
        horas_promedio DOUBLE COMMENT 'Horas en puerto promedio por buque',
        horas_totales DOUBLE, _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_puerto_dia PRIMARY KEY (fecha, wpi_id)""",
    T_BUQUES_PUERTO: """fecha DATE NOT NULL, wpi_id INT NOT NULL, nombre_puerto STRING,
        MMSI_Real STRING NOT NULL, MMSI STRING, nombre_buque STRING, categoria STRING,
        primera_ts TIMESTAMP, ultima_ts TIMESTAMP, horas_en_puerto DOUBLE, arribos BIGINT,
        presente_al_inicio BOOLEAN, _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_buques_en_puerto_dia PRIMARY KEY (fecha, wpi_id, MMSI_Real)""",
    T_ANOM_BUQUE: """fecha DATE NOT NULL, MMSI STRING NOT NULL, tipo STRING NOT NULL,
        nombre_buque STRING, eventos BIGINT, valor_max DOUBLE,
        ultima_lat DOUBLE, ultima_lon DOUBLE, ultimo_detalle STRING, _version_silver TIMESTAMP NOT NULL,
        CONSTRAINT pk_anomalias_buque_dia PRIMARY KEY (fecha, MMSI, tipo)""",
}
for tabla, columnas in contratos.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {tabla} ({columnas}) CLUSTER BY (fecha)")

FECHAS, VERSIONES = comun.fechas_pendientes(spark, T_METRICAS, TABLAS_SERVICIO, RECALCULAR_TODO)
print(f"Fechas por recalcular: {[str(d) for d in FECHAS] or 'ninguna'}")

In [0]:
if FECHAS:
    en = lambda t: spark.table(t).filter(f.col("fecha").isin(FECHAS))
    ver = comun.df_versiones(spark, FECHAS, VERSIONES)
    categorias = spark.table(T_DIM_TIPO).select(f.col("vessel_type").alias("VesselType"), "categoria")
    nombres = en(T_SILVER).groupBy("fecha", "MMSI").agg(f.max_by("VesselName", "BaseDateTime").alias("nombre_buque"))

    # trafico_celda_dia: buques distintos REALES del día (no la suma de horas)
    wkt = dbf.h3_centeraswkt("h3_celda")
    punto = r"POINT\s*\(\s*(\S+)\s+(\S+)\s*\)"
    celda = (
        en(T_PUNTOS).withColumn("h3_celda", dbf.h3_toparent("H3_Micro", config.RES_MAPA))
        .groupBy("fecha", "h3_celda")
        .agg(f.count_distinct("MMSI").alias("buques"), f.count("*").alias("posiciones"), f.avg("SOG").alias("sog_promedio"))
        .withColumn("lon", f.regexp_extract(wkt, punto, 1).cast("double"))
        .withColumn("lat", f.regexp_extract(wkt, punto, 2).cast("double"))
        .withColumn("intensidad", f.log10("posiciones"))
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, celda, T_CELDA, FECHAS)

    # trafico_hora: buques distintos por hora en todo el país (sumar celdas contaría doble)
    hora = (
        en(T_PUNTOS).join(f.broadcast(categorias), "VesselType", "left")
        .groupBy("fecha", f.hour("BaseDateTime").alias("hora"), f.coalesce("categoria", f.lit("Desconocido")).alias("categoria"))
        .agg(f.count_distinct("MMSI").alias("buques"), f.count("*").alias("posiciones"))
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, hora, T_HORA, FECHAS)

    # puerto_dia
    puertos = spark.table(T_DIM_PUERTO).select("wpi_id", "lat", "lon")
    puerto_dia = (
        en(T_PRESENCIA).groupBy("fecha", "wpi_id", "nombre_puerto", "pais")
        .agg(
            f.sum("arribos").alias("arribos"),
            f.count_distinct("MMSI_Real").alias("buques_presentes"),
            f.avg("horas_en_puerto").alias("horas_promedio"),
            f.sum("horas_en_puerto").alias("horas_totales"),
        )
        .join(f.broadcast(puertos), "wpi_id", "left")
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, puerto_dia, T_PUERTO_DIA, FECHAS)

    # buques_en_puerto_dia
    buques_puerto = (
        en(T_PRESENCIA).join(nombres, ["fecha", "MMSI"], "left")
        .join(f.broadcast(categorias), "VesselType", "left")
        .select(
            "fecha", "wpi_id", "nombre_puerto", "MMSI_Real", "MMSI", "nombre_buque",
            f.coalesce("categoria", f.lit("Desconocido")).alias("categoria"),
            "primera_ts", "ultima_ts", "horas_en_puerto", "arribos", "presente_al_inicio",
        )
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, buques_puerto, T_BUQUES_PUERTO, FECHAS)

    # anomalias_buque_dia
    anom_buque = (
        en(T_ANOMALIAS).groupBy("fecha", "MMSI", "tipo")
        .agg(
            f.count("*").alias("eventos"), f.max("valor").alias("valor_max"),
            f.max_by("lat", "ts").alias("ultima_lat"), f.max_by("lon", "ts").alias("ultima_lon"),
            f.max_by("detalle", "ts").alias("ultimo_detalle"),
        )
        .join(nombres, ["fecha", "MMSI"], "left")
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, anom_buque, T_ANOM_BUQUE, FECHAS)

    # kpi_dia (al final: usa puerto_dia)
    m = (
        en(T_METRICAS).groupBy("fecha")
        .pivot("metrica", ["filas_recibidas", "filas_cuarentena"]).agg(f.first("valor"))
    )
    kpi = (
        en(T_DIARIAS).groupBy("fecha").agg(
            f.count_distinct("MMSI").alias("buques_activos"), f.count("*").alias("viajes"),
            f.sum("distancia_nm").alias("millas"))
        .join(en(T_PUERTO_DIA).groupBy("fecha").agg(f.sum("arribos").alias("arribos")), "fecha", "left")
        .join(en(T_PRESENCIA).groupBy("fecha").agg(f.count_distinct("MMSI").alias("buques_en_puerto")), "fecha", "left")
        .join(en(T_ANOMALIAS).groupBy("fecha").agg(f.count_distinct("MMSI").alias("buques_con_anomalias")), "fecha", "left")
        .join(m, "fecha", "left")
        .withColumn("pct_cuarentena", f.col("filas_cuarentena") * 100.0 / f.col("filas_recibidas"))
        .join(ver, "fecha")
    )
    comun.escribir_fechas(spark, kpi, T_KPI, FECHAS)

In [0]:
spark.table(T_KPI).orderBy("fecha").show(truncate=False)

cuadre = (
    spark.table(T_CELDA).groupBy("fecha").agg(f.sum("posiciones").alias("posiciones_mapa"))
    .join(spark.table(T_PUNTOS).groupBy("fecha").agg(f.count("*").alias("posiciones_oro")), "fecha")
    .join(spark.table(T_PUERTO_DIA).groupBy("fecha").agg(f.sum("arribos").alias("arribos_servicio")), "fecha")
    .join(spark.table(T_PRESENCIA).groupBy("fecha").agg(f.sum("arribos").alias("arribos_oro")), "fecha")
    .orderBy("fecha")
)
cuadre.show(truncate=False)
malas = cuadre.filter(
    (f.col("posiciones_mapa") != f.col("posiciones_oro")) | (f.col("arribos_servicio") != f.col("arribos_oro"))
).count()
assert malas == 0, "Las tablas de servicio no cuadran con oro"
print("Servicio cuadra con oro.")

In [0]:
spark.sql(f"""
SELECT r AS resolucion, COUNT(*) AS celdas, COUNT_IF(n >= 20) AS celdas_con_20_o_mas
FROM (
  SELECT r, h3_toparent(H3_Micro, r) AS celda, COUNT(*) AS n
  FROM {T_PUNTOS} CROSS JOIN (SELECT explode(array(5, 6, 7)) AS r)
  WHERE fecha = DATE'2023-06-01'
  GROUP BY r, celda
)
GROUP BY r ORDER BY r
""").show()